# Instrument-Agnostic Automatic Music Transcription (Colab Inference)

This notebook allows you to run inference using the [Instrument-Agnostic AMT](https://github.com/anime-song/instrument-agnostic-amt) model.
It will automatically download the pre-trained model from Hugging Face.

## 1. Setup Environment

In [ ]:
# @title Install dependencies
!git clone https://github.com/haveyouwantto/instrument-agnostic-amt.git
%cd instrument-agnostic-amt
!uv pip install -r requirements.txt
!uv pip install yt-dlp stem-splitter # Optional: yt-dlp for downloading from YouTube

!uv pip install git+https://github.com/xavriley/ADTOF-pytorch.git


## 2. Prepare Audio

You can either upload a file or download one from a URL (e.g., YouTube).

In [ ]:
# @title Upload audio file
from google.colab import files
import os

uploaded = files.upload()
audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {audio_path}")

## 3. Run Inference

In [ ]:
# @title Run Transcription
!python infer.py --audio "{audio_path}"

import os
midi_path = os.path.splitext(audio_path)[0] + ".mid"
if os.path.exists(midi_path):
    print(f"Success! MIDI saved to: {midi_path}")
else:
    print("Error: MIDI file was not generated.")

## 4. Download Results

In [ ]:
# @title Download MIDI file
if os.path.exists(midi_path):
    files.download(midi_path)
else:
    print("No MIDI file to download.")

## 5. Optional: Stem Separation -> Transcribe Each Stem -> Merge

This section mirrors the `batch_process_unlabeled.py` flow inside Colab.
It separates the uploaded song into stems, transcribes each non-drum stem, and merges the per-stem MIDI files into one result.


In [ ]:
# @title Run stem-separated transcription
OUTPUT_ROOT = "colab_outputs"  # @param {type:"string"}
WINDOW_BATCH_SIZE = 4  # @param {type:"integer"}
MAX_MIDI_MELODIC_INSTRUMENTS = 15  # @param {type:"integer"}
CLEANUP_SEPARATED_STEMS = False  # @param {type:"boolean"}
MERGE_ONSET_MS = 50.0  # @param {type:"number"}
USE_ADTOF_TRANSCRIBER = True  # @param {type:"boolean"}
USE_STFT_VOCALIZER = False  # @param {type:"boolean"}
USE_TRANSKUN = True # @param {type:"boolean"}


from separate_helper import run_stem_separated_transcription

if "audio_path" not in globals():
    raise RuntimeError("Please upload an audio file first.")

stem_pipeline_result = run_stem_separated_transcription(
    audio_path,
    checkpoint_path=None,
    output_root=OUTPUT_ROOT,
    window_batch_size=WINDOW_BATCH_SIZE,
    max_midi_melodic_instruments=MAX_MIDI_MELODIC_INSTRUMENTS,
    cleanup_separated_stems=CLEANUP_SEPARATED_STEMS,
    merge_onset_ms=MERGE_ONSET_MS,
    use_adtof_transcriber=USE_ADTOF_TRANSCRIBER,
    use_stft_vocalizer=USE_STFT_VOCALIZER,
    use_transkun=USE_TRANSKUN
)
stem_pipeline_result


In [ ]:
# @title Download stem-separated results
from google.colab import files
from pathlib import Path
import shutil

if "stem_pipeline_result" not in globals():
    print("Run the stem-separated transcription cell first.")
else:
    merged_midi_path = Path(stem_pipeline_result["merged_midi_path"])
    stem_midi_dir = Path(stem_pipeline_result["stem_midi_dir"])
    zip_base = stem_midi_dir.parent / f"{stem_midi_dir.name}"
    zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=stem_midi_dir))

    print(f"Downloading merged MIDI: {merged_midi_path}")
    files.download(str(merged_midi_path))
